# Credit score model (LightGBM regression)

**Fixed version.** The previous version of this notebook applied `sklearn.preprocessing.Normalizer` row-wise across `[Income, LoanAmount, CreditScore, InterestRate, DTIRatio]` *together*, which meant the target (`CreditScore`) was baked into the normalization of its own input features — label leakage. `mlPredictor.py` had to work around this at inference time with a `CreditScore=0` placeholder, which only stayed accurate because `Income`'s magnitude dwarfs `CreditScore`'s in the L2 norm — an accident of scale, not a real fix.

This version:
- Splits train/test **before** fitting any scaler.
- Scales **input features only** with a column-wise `StandardScaler` fit on the train split only. `CreditScore` (the target) is never part of that fit or transform.
- Declares `LoanPurpose` as categorical to LightGBM instead of letting it be read as an ordinal integer.
- Uses a real held-out validation split for early stopping, and evaluates on the test set exactly once at the end.
- Uses `rmse` as the training metric (`"r2"` is not a native LightGBM metric name and was silently ignored before).
- Exports the fitted scaler's actual `mean_`/`scale_` into the preprocessing artifact, replacing the row-norm inversion hack in `mlPredictor.py` with a plain z-score transform — the same shape already used for the PD model's `scaler.mean`/`scaler.std`.

**Not yet run** — `Loan_default.csv` isn't in the repo. Drop it next to this notebook (or update the path below) and run top-to-bottom.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
import lightgbm as lgb


In [ ]:
df = pd.read_csv('Loan_default.csv')
df = df[['LoanAmount', 'Income', 'InterestRate', 'DTIRatio', 'Age',
         'LoanPurpose', 'HasMortgage', 'HasDependents', 'CreditScore']]


In [ ]:
df['HasMortgage'] = df['HasMortgage'].replace(['Yes', 'No'], [1, 0]).astype(int)
df['HasDependents'] = df['HasDependents'].replace(['Yes', 'No'], [1, 0]).astype(int)
df['LoanPurpose'] = df['LoanPurpose'].replace(
    ['Business', 'Home', 'Education', 'Other', 'Auto'], [0, 1, 2, 3, 4]
).astype('category')


## Split before scaling
Everything downstream (scaler fit, engineered features) is derived only from the train split from this point on; the test split stays untouched until the single final evaluation cell.

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.copy()
test_df = test_df.copy()


## Scale input features only (never the target)
Column-wise `StandardScaler`, fit on `train_df` only. `CreditScore` is deliberately excluded from `numeric_features` — it is the regression target and must never appear on the right-hand side of a transform applied to the inputs.

In [ ]:
numeric_features = ['Income', 'LoanAmount', 'InterestRate', 'DTIRatio']

scaler = StandardScaler()
train_df[numeric_features] = scaler.fit_transform(train_df[numeric_features])
test_df[numeric_features] = scaler.transform(test_df[numeric_features])


In [ ]:
for part in (train_df, test_df):
    part['DTIRatio*InterestRate'] = np.log(
        np.abs(part['DTIRatio'] * part['InterestRate']) + 1e-10
    )
    part['LoanAmount/Income'] = np.abs(
        np.log(np.abs(part['LoanAmount'] / part['Income']) + 1e-10)
    ) ** 0.30


In [ ]:
feature_cols = ['Age', 'LoanPurpose', 'HasMortgage', 'HasDependents',
                'DTIRatio*InterestRate', 'LoanAmount/Income']

X_train_full, y_train_full = train_df[feature_cols], train_df['CreditScore']
X_test, y_test = test_df[feature_cols], test_df['CreditScore']

# Held-out validation split from train only, for early stopping.
# The test set above is never touched until the final evaluation cell.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42
)


In [ ]:
train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=['LoanPurpose'])
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data,
                        categorical_feature=['LoanPurpose'])

params = {
    "boosting_type": "goss",
    "objective": "regression",
    "metric": "rmse",  # "r2" is not a native LightGBM metric; it was silently ignored before.
    "learning_rate": 0.05,
    "num_leaves": 64,
    "max_depth": -1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "verbose": -1,
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[val_data],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(50)],
)


## Final evaluation — test set touched exactly once, here

In [ ]:
y_pred = model.predict(X_test, num_iteration=model.best_iteration)

print(f"Test R\u00b2:  {r2_score(y_test, y_pred):.4f}")
print(f"Test MAE: {mean_absolute_error(y_test, y_pred):.2f} credit-score points")


## Save model + preprocessing artifact
The scaler's real fitted `mean_`/`scale_` replace the guessed/reconstructed values that were previously hand-set in `ml_artifacts/preprocessing_v1.json` (there was never a real fitted scaler to export before, since the training data wasn't available). `mlPredictor.py`'s credit-score serving path needs a matching update once this actually runs: drop the `row_norm`/`denormalize_credit_score` inversion entirely (the target is no longer scaled, so the model's raw output *is* the credit score, only needing the existing [300, 850] clip) and scale the four numeric inputs with the plain z-score `scale_features()` helper that already exists for the PD model.

In [ ]:
import json
from pathlib import Path

out_dir = Path('../models')
out_dir.mkdir(exist_ok=True)
model.save_model(str(out_dir / 'creditscore_model.txt'))

artifact_path = Path('../ml_artifacts/preprocessing_v2.json')
existing = json.loads(artifact_path.read_text()) if artifact_path.exists() else {}

existing['version'] = 'v2'
existing.setdefault('credit_score', {})
existing['credit_score']['numerical_impute'] = {
    'age': 40, 'income': 60000, 'loanAmount': 20000,
    'loanRate': 10, 'existingDebtPayment': 500,
}
existing['credit_score']['categorical_mapping'] = {
    'loanPurpose': {
        'Business': 0, 'Home': 1, 'Education': 2,
        'Other': 3, 'Others': 3, 'Auto': 4, 'Automobile': 4,
    }
}
existing['credit_score']['scaler'] = {
    'features': numeric_features,
    'mean': dict(zip(numeric_features, scaler.mean_.tolist())),
    'std': dict(zip(numeric_features, scaler.scale_.tolist())),
}
existing['credit_score']['target_is_scaled'] = False

artifact_path.write_text(json.dumps(existing, indent=2))
print(f"Wrote {artifact_path}")
